# Sales Price-Volume Analysis: Python Implementation

This notebook imports the synthetic sales dataset and implements the price-impact methodology from Section 4 of the overview. Calculate effects for individual customer-product pairs, then query or summarize them by product, customer, and overall.

Keep this notebook and `sales_variance_dataset.csv` in the same working folder.

## 1. Import the Dataset

We use pandas to read the CSV into a DataFrame named `df_sales`. Each row represents one customer-product combination in one year. Prices and sales are in EUR; there are no separate fixed charges.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("sales_variance_dataset.csv")
df_sales = pd.read_csv(data_path, encoding="utf-8-sig")

print(f"Loaded {len(df_sales):,} rows and {len(df_sales.columns)} columns.")

Loaded 1,000 rows and 6 columns.


## 2. Preview the Data

The table contains the requested columns: `Year`, `Customer`, `Product`, `Quantity`, `Price`, and `Sales`. The preview and counts below confirm that the file was loaded.

In [2]:
df_sales.head(10)

,Year,Customer,Product,Quantity,Price,Sales
0,2025,Customer_001,Product_A,272,4.60,1251.20
1,2025,Customer_001,Product_B,243,7.58,1841.94
2,2025,Customer_001,Product_C,76,11.04,839.04
3,2025,Customer_001,Product_D,406,16.89,6857.34
4,2025,Customer_001,Product_E,363,23.00,8349.00
5,2025,Customer_002,Product_A,479,5.20,2490.80
6,2025,Customer_002,Product_B,351,8.16,2864.16
7,2025,Customer_002,Product_C,916,12.36,11321.76
8,2025,Customer_002,Product_D,874,18.18,15889.32
9,2025,Customer_002,Product_E,943,25.50,24046.50


In [3]:
print("Years:", sorted(df_sales["Year"].unique().tolist()))
print("Customers:", df_sales["Customer"].nunique())
print("Products:", df_sales["Product"].nunique())
print("Rows per year:")
print(df_sales.groupby("Year").size().to_string())

Years: [2025, 2026]
Customers: 100
Products: 5
Rows per year:
Year
2025    500
2026    500


## 3. Prepare Comparable Customer-Product Pairs

We match the same customer and product in 2025 and 2026. This synthetic dataset assumes established base business for every pair. The checks below require one positive-quantity record per pair per year; new or lost relationships must be handled separately rather than silently dropped.

In [4]:
import numpy as np
from IPython.display import display

previous_year = 2025
current_year = 2026
keys = ['Customer', 'Product']

# The synthetic dataset contains comparable base business only.
period_data = df_sales[df_sales['Year'].isin([previous_year, current_year])].copy()
required = ['Year', 'Customer', 'Product', 'Quantity', 'Price', 'Sales']
if previous_year == current_year:
    raise ValueError('Choose two different comparison years.')
if period_data[required].isna().any().any():
    raise ValueError('Required fields contain missing values.')
if period_data.duplicated(['Year'] + keys).any():
    raise ValueError('Expected one row per year/customer/product. Aggregate transactions first.')
if (period_data[['Quantity', 'Price']] <= 0).any().any():
    raise ValueError('This example requires positive quantities and prices.')
if not np.isfinite(period_data[['Quantity', 'Price', 'Sales']].to_numpy()).all():
    raise ValueError('Sales data must contain finite numbers.')
if not np.allclose(period_data['Sales'], period_data['Quantity'] * period_data['Price'],
                   rtol=0, atol=0.005):
    raise ValueError('Sales must equal Quantity x Price; fixed charges are outside this dataset.')

def select_period(year, suffix):
    period = period_data.loc[period_data['Year'].eq(year), keys + ['Quantity', 'Price', 'Sales']]
    if period.empty:
        raise ValueError(f'No sales data found for {year}.')
    return period.rename(columns={name: f'{name}_{suffix}' for name in ['Quantity', 'Price', 'Sales']})

previous = select_period(previous_year, 'Previous')
current = select_period(current_year, 'Current')
comparison = previous.merge(current, on=keys, how='outer', validate='one_to_one', indicator=True)
if not comparison['_merge'].eq('both').all():
    raise ValueError('Unmatched customer-product pairs found. Classify new/lost business separately.')
comparison = comparison.drop(columns='_merge').sort_values(keys).reset_index(drop=True)
print(f'Comparing {previous_year} with {current_year}: {len(comparison)} comparable pairs.')


Comparing 2025 with 2026: 500 comparable pairs.


## 4. Calculate the Price Impact

From Section 4 of the overview, the price effect is $(P_2-P_1)V_1+(F_2-F_1)$. There are no fixed charges in this dataset, so for each customer-product pair:

$$\text{Price Impact}_i=(P_{2026,i}-P_{2025,i})Q_{2025,i}.$$

We calculate this at the lowest comparison level first. A product result sums its contributions across customers; a customer result sums contributions across the products bought. Company-wide average price changes would also reflect mix and should not replace this calculation.

A positive contribution means prices increased revenue on the previous quantity; a negative contribution means prices reduced it. Monetary values use the dataset's two-decimal pricing precision.

In [5]:
# Change the price only, keeping previous-year quantity constant.
comparison['Price_Change'] = (comparison['Price_Current'] - comparison['Price_Previous']).round(2)
comparison['Price_Impact'] = (comparison['Price_Change'] * comparison['Quantity_Previous']).round(2)

# Audit against the intermediate scenario S* = P_current * V_previous.
comparison['Repriced_Previous_Sales'] = (
    comparison['Price_Current'] * comparison['Quantity_Previous']
).round(2)
assert np.allclose(
    comparison['Price_Impact'],
    comparison['Repriced_Previous_Sales'] - comparison['Sales_Previous'],
    rtol=0, atol=0.005
)
detail_columns = ['Customer', 'Product', 'Quantity_Previous', 'Price_Previous',
                  'Price_Current', 'Price_Change', 'Price_Impact']
money_columns = ['Price_Previous', 'Price_Current', 'Price_Change', 'Price_Impact']

def format_details(table):
    return table.style.format({column: '{:,.2f}' for column in money_columns})

display(format_details(comparison[detail_columns].head(10)))


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_001,Product_B,243,7.58,6.97,-0.61,-148.23
2,Customer_001,Product_C,76,11.04,11.81,0.77,58.52
3,Customer_001,Product_D,406,16.89,15.88,-1.01,-410.06
4,Customer_001,Product_E,363,23.00,26.91,3.91,"1,419.33"
5,Customer_002,Product_A,479,5.20,5.36,0.16,76.64
6,Customer_002,Product_B,351,8.16,7.51,-0.65,-228.15
7,Customer_002,Product_C,916,12.36,14.58,2.22,"2,033.52"
8,Customer_002,Product_D,874,18.18,17.45,-0.73,-638.02
9,Customer_002,Product_E,943,25.50,27.29,1.79,"1,687.97"


## 5. Summarize by Product, Customer, and Overall

The same pair-level contributions are grouped in two ways. Each grouping must add to the same total price impact. Amounts are in EUR; these are revenue effects, not profit contributions.

In [6]:
product_summary = comparison.groupby('Product', as_index=False)['Price_Impact'].sum()
product_summary['Price_Impact'] = product_summary['Price_Impact'].round(2)
customer_summary = comparison.groupby('Customer', as_index=False)['Price_Impact'].sum()
customer_summary['Price_Impact'] = customer_summary['Price_Impact'].round(2)
total_price_impact = round(comparison['Price_Impact'].sum(), 2)

print(f'Total price impact ({previous_year} to {current_year}): EUR {total_price_impact:,.2f}')
print('All products:')
display(product_summary.style.format({'Price_Impact': '{:,.2f}'}))
print('First 10 customers (the full table is stored in customer_summary):')
display(customer_summary.head(10).style.format({'Price_Impact': '{:,.2f}'}))


Total price impact (2025 to 2026): EUR 143,231.87
All products:


,Product,Price_Impact
0,Product_A,"5,706.92"
1,Product_B,"-8,227.50"
2,Product_C,"51,566.61"
3,Product_D,"4,994.35"
4,Product_E,"89,191.49"


First 10 customers (the full table is stored in customer_summary):


,Customer,Price_Impact
0,Customer_001,919.56
1,Customer_002,"2,931.96"
2,Customer_003,"2,303.94"
3,Customer_004,210.13
4,Customer_005,"1,335.39"
5,Customer_006,"1,276.68"
6,Customer_007,"1,670.35"
7,Customer_008,"1,623.64"
8,Customer_009,-256.19
9,Customer_010,"1,290.92"


## 6. Query a Product or Customer

Use `price_impact(product="Product_A")` for a product total, `price_impact(customer="Customer_001")` for a customer total, or `price_impact()` for the overall total. To inspect the contributions behind the number, use `show_price_impact(...)`. Both filters can be supplied to inspect one pair. Unknown identifiers produce a clear error rather than a misleading zero.

In [7]:
def price_impact_details(product=None, customer=None):
    # Filter contributions already calculated at customer-product level.
    # Identifiers accept surrounding spaces and any letter case.
    selected = comparison
    for column, value in [('Product', product), ('Customer', customer)]:
        if value is not None:
            identifier = str(value).strip().casefold()
            valid_names = comparison[column].drop_duplicates().tolist()
            if identifier not in {str(name).casefold() for name in valid_names}:
                examples = ', '.join(valid_names[:5])
                raise ValueError(f'Unknown {column.lower()}: {value!r}. Examples: {examples}')
            selected = selected[selected[column].str.casefold().eq(identifier)]
    if selected.empty:
        raise ValueError('No comparable sales found for the selected customer/product.')
    return selected[detail_columns].copy().reset_index(drop=True)


def price_impact(product=None, customer=None):
    # No filters = total business. Both filters = one customer-product pair.
    details = price_impact_details(product=product, customer=customer)
    return round(details['Price_Impact'].sum(), 2)


def show_price_impact(product=None, customer=None):
    amount = price_impact(product=product, customer=customer)
    scope = ', '.join(str(value).strip() for value in [product, customer] if value is not None)
    print(f'Price impact for {scope or "all customers and products"}: EUR {amount:,.2f}')
    if product is None and customer is None:
        display(product_summary.style.format({'Price_Impact': '{:,.2f}'}))
    else:
        display(format_details(price_impact_details(product=product, customer=customer)))


### 6.1. Enter a Product

Change `selected_product` below and run the cell. The result sums the chosen product's price contributions across all customers. The table shows how each customer contributes.

In [8]:
selected_product = "Product_A"
show_price_impact(product=selected_product)

Price impact for Product_A: EUR 5,706.92


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_002,Product_A,479,5.20,5.36,0.16,76.64
2,Customer_003,Product_A,297,4.44,4.44,0.00,0.00
3,Customer_004,Product_A,141,4.90,5.15,0.25,35.25
4,Customer_005,Product_A,283,4.90,4.90,0.00,0.00
5,Customer_006,Product_A,403,4.61,5.12,0.51,205.53
6,Customer_007,Product_A,762,4.69,4.69,0.00,0.00
7,Customer_008,Product_A,485,4.38,4.38,0.00,0.00
8,Customer_009,Product_A,558,4.51,4.51,0.00,0.00
9,Customer_010,Product_A,944,4.40,4.58,0.18,169.92


### 6.2. Enter a Customer

Change `selected_customer` below and run the cell. The five product contributions are displayed, and their sum is the customer's price impact.

In [9]:
selected_customer = "Customer_001"
show_price_impact(customer=selected_customer)

Price impact for Customer_001: EUR 919.56


,Customer,Product,Quantity_Previous,Price_Previous,Price_Current,Price_Change,Price_Impact
0,Customer_001,Product_A,272,4.60,4.60,0.00,0.00
1,Customer_001,Product_B,243,7.58,6.97,-0.61,-148.23
2,Customer_001,Product_C,76,11.04,11.81,0.77,58.52
3,Customer_001,Product_D,406,16.89,15.88,-1.01,-410.06
4,Customer_001,Product_E,363,23.00,26.91,3.91,"1,419.33"


### 6.3. Overall Price Impact

No filter sums all 500 customer-product contributions.

In [10]:
show_price_impact()

Price impact for all customers and products: EUR 143,231.87


,Product,Price_Impact
0,Product_A,"5,706.92"
1,Product_B,"-8,227.50"
2,Product_C,"51,566.61"
3,Product_D,"4,994.35"
4,Product_E,"89,191.49"


## 7. Interactive Lookup

After running all cells, choose **Product**, **Customer**, or **Total**. Type or select the identifier in the relevant field, then click **Calculate price impact**. These controls require a Jupyter interface with widget support; the editable example cells above provide the same calculations without widgets.

In [11]:
# Optional interactive controls. The functions above work without ipywidgets.
try:
    import ipywidgets as widgets
except ImportError:
    print('Interactive controls are unavailable. Use show_price_impact(product="Product_A") '
          'or show_price_impact(customer="Customer_001") instead.')
else:
    scope_input = widgets.Dropdown(options=['Product', 'Customer', 'Total'],
                                   value='Product', description='View:')
    product_input = widgets.Combobox(options=tuple(sorted(comparison['Product'].unique())),
                                   value='Product_A', description='Product:')
    customer_input = widgets.Combobox(options=tuple(sorted(comparison['Customer'].unique())),
                                    value='Customer_001', description='Customer:')
    calculate_button = widgets.Button(description='Calculate price impact', button_style='info',
                                      layout=widgets.Layout(width='210px'))
    query_output = widgets.Output()

    def update_inputs(change=None):
        product_input.disabled = scope_input.value != 'Product'
        customer_input.disabled = scope_input.value != 'Customer'

    def calculate_selected(button=None):
        with query_output:
            query_output.clear_output(wait=True)
            try:
                if scope_input.value == 'Product':
                    show_price_impact(product=product_input.value)
                elif scope_input.value == 'Customer':
                    show_price_impact(customer=customer_input.value)
                else:
                    show_price_impact()
            except ValueError as error:
                print(error)

    scope_input.observe(update_inputs, names='value')
    calculate_button.on_click(calculate_selected)
    update_inputs()
    display(widgets.VBox([scope_input, product_input, customer_input,
                         calculate_button, query_output]))


## 8. Reconcile the Results

Summing by customer and summing by product must give the same overall price effect. The checks below verify the totals and each lookup against its summary table.

In [12]:
# Every view must add up to the same overall price effect.
assert np.isclose(product_summary['Price_Impact'].sum(), total_price_impact, rtol=0, atol=0.005)
assert np.isclose(customer_summary['Price_Impact'].sum(), total_price_impact, rtol=0, atol=0.005)
assert all(np.isclose(price_impact(product=product), amount, rtol=0, atol=0.005)
           for product, amount in product_summary.itertuples(index=False, name=None))
assert all(np.isclose(price_impact(customer=customer), amount, rtol=0, atol=0.005)
           for customer, amount in customer_summary.itertuples(index=False, name=None))
print('Verified: product totals = customer totals = the overall price impact.')


Verified: product totals = customer totals = the overall price impact.
